# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [2]:
from pathlib import Path
import csv

input_path = Path("assets/inference_samples.csv")

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

Fichero: assets\inference_samples.csv
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Filas de ejemplo: 5


## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Tener un modelo distinto al validado |
| Versión del esquema | **En el manifest.json** | Contratos de entrada/salida incompatibles |
| Orden y nombres de features | **En el manifest.json** | Features erróneas o desordenadas sin error visible |
| Versión del preprocesado | **En el manifest.json** | Para que el preprocesado con el que fue entrenado el modelo sea el mismo que apliquemos nosotros |
| Etiquetas de salida permitidas | **En el manifest.json** | Clases inesperadas o mal mapeadas |
| Versión del modelo | **En el manifest.json** | Falta de trazabilidad |
| Tipo de estimador | **En el manifest.json** | Cargar el artefacto equivocado |

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
    "schema_version": "v1.0",
    "model_version": "wine-quality-rf-v1",
    "preprocess_version": "v1.0",
    "features_name": [
        "fixed_acidity",
        "volatile_acidity",
        "citric_acid",
        "residual_sugar",
        "chlorides",
        "free_sulfur_dioxide",
        "total_sulfur_dioxide",
        "density",
        "ph",
        "sulphates",
        "alcohol"
    ],
    "output_labels": [
        "acceptable",
        "excellent",
        "needs_review"
    ],
    "estimator_type": {
        "RandomForestClassifier": {
            "class_weight": "balanced_subsample",
            "max_depth": 8,
            "min_samples_leaf": 2,
            "n_estimators": 200,
            "random_state": 42
        }
    }
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [6]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]



# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

estimator_type={
"RandomForestClassifier": {
        "class_weight": "balanced_subsample",
        "max_depth": 8,
        "min_samples_leaf": 2,
        "n_estimators": 200,
        "random_state": 42
    }
}
manifest_draft = {
    "schema_version": "v1.0",
    "model_version": legacy_model_version,
    "preprocessing_version": "v1.0",
    "feature_names": feature_names,
    "output_labels": ["low", "medium", "high"],
    "estimator_type": estimator_type,
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

wrong_manifest_draft = {
    "schema_version": "v1.0",
    "model_version": legacy_model_version,
    "preprocessing_version": "v2.0",
    "feature_names": feature_names,
    "output_labels": ["low", "medium", "high"],
    "estimator_type": estimator_type,
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

{
  "schema_version": "v1.0",
  "model_version": "wine-quality-rf-demo-v1",
  "preprocessing_version": "v1.0",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "low",
    "medium",
    "high"
  ],
  "estimator_type": {
    "RandomForestClassifier": {
      "class_weight": "balanced_subsample",
      "max_depth": 8,
      "min_samples_leaf": 2,
      "n_estimators": 200,
      "random_state": 42
    }
  }
}

La estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.


## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | Validación de entrada | "Orden de features no coincide con el manifest" | No |
| Cambia `preprocessing_version` | Carga del artefacto | "preprocessing_version incompatible" | No |
| El modelo devuelve `unknown` | Validación de salida | "Etiqueta 'unknown' no permitida" | No |
| La petición incluye `ph = 99` | Validación de entrada | "ph=99 fuera de rango [5,20 ]" | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
- ¿Aceptarás campos desconocidos en el manifiesto?
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.

## 5. Práctica 4 — Preparar la implementación de la segunda clase

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
Obligatorios: `model_version`, `schema_version`, `preprocessing_version`, `estimator_type`, `feature_names` y `allowed_labels`.

- ¿Aceptarás campos desconocidos en el manifiesto?
No. Se rechazan (`extra="forbid"`).

- ¿Validarás el manifiesto antes de cargar el objeto serializado?
Sí. Así se falla antes y no se deserializa el artefacto.

- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
Solo después de validar todas las entradas, para que no quede un CSV parcial.

- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
Un mensaje con el campo, el valor recibido y el esperado (`Error: fila 2, ph=99 fuera de rango [5, 20]`) con un código de salida distinto de 0.

- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?
Falla todo y no se escribe nada. Como dije antes, no se quiere un csv parcial, el error indica la fila 2.